# GameTheory-06e : Transparence des programmes et issue du Dilemme du prisonnier one-shot

Quand l'agent adverse peut **lire** votre strategie, l'equilibre de Nash `(D,D)` du PD classique
cesse d'etre l'unique verdict : la transparence du programme change l'issue. Ce notebook
explore cette frontiere entre jeu sous forme normale et jeu sous forme extensive ou les
joueurs sont des **programmes** (heuristiques `_toy`, cf. header §0) bornes et totaux.

References : Shoham & Leyton-Brown (2009) §3.4.2 ; Osborne (2004) §3.2 ; Axelrod & Hamilton (*Science*, 1981).


## 0. Sources primaires et conventions de nommage

**Sources publiées citées dans ce notebook** :

- **Shoham & Leyton-Brown**, *Multiagent Systems: Algorithmic, Game-Theoretic, and Logical Foundations* (Cambridge University Press, 2009), chapitre **§3.4.2 « Iterated Dominance and the Prisoner's Dilemma »**. C'est la référence pour la paramétrisation canonique du Dilemme du prisonnier `(T=5, R=3, P=1, S=0)` et la condition `2R > T + S` qui distingue un PD **strict** d'un jeu à alternance (cf. exercice 3 de Shoham-Leyton-Brown §3.4).
- **Osborne**, *An Introduction to Game Theory* (Oxford University Press, 2004), chapitre **§3.2 « Strictly Dominant Strategies »**. La dominance stricte de `D` sur `C` pour les deux joueurs — peu importe l'action de l'adversaire — établit `(D, D)` comme équilibre de Nash **unique** en stratégies pures.
- **Axelrod & Hamilton**, « The Evolution of Cooperation » (*Science*, vol. 211, n° 4489, 1981, pp. 1390-1396). Cité pour `TitForTat` (Axelrod, 1980) dans le contexte des jeux répétés. `TitForTat` n'est **pas** implémenté dans le moteur du présent notebook — il figure en exercice 1 pour situer la transparence des programmes par rapport à la littérature des stratégies itérées.
- **Critch, Dennis & Russell**, « Reflective Probabilistic Programs » (2022, [arXiv:2208.07006](https://arxiv.org/abs/2208.07006)), **§3 « Inspection games »** et **Open Problem 3** (p. 18). C'est la référence canonique pour les définitions de `DUPOC(k)` (*Do-Unto-Other-Player with Probability k of Cooperation*) et `CUPOD(k)` (*Cooperate-Unless-Player-is-Defecting*) dans le cadre de l'inspection mutuelle de programmes. **Open Problem 3** pose la conjecture sur le comportement asymptotique de `DUPOC(k)` vs `CUPOD(k)` au-delà de quelques itérations ; ce notebook l'illustre mais ne la résout pas (cf. cellule 26).

**Conventions de nommage** :

- Les bots implémentés dans ce notebook sont des **heuristiques ad hoc** (locales au notebook), suffixés `_toy` : `CooperateBot_toy`, `DefectBot_toy`, `FairBot_toy`, `CUPOD_toy`, `PrudentBot_toy`. Ce suffixe marque la frontière entre la pédagogie locale et la littérature publiée. Aucune des stratégies `_toy` n'est une reproduction d'une stratégie publiée (la ressemblance FairBot/DUPOC et CUPOD_toy/CUPOD est **intentionnelle**, pas une citation : `DUPOC` et `CUPOD` sont des concepts théoriques définis par Critch-Dennis-Russell 2022 §3, et aucune instance de ces stratégies n'est reproduite ici).
- `TitForTat` (Axelrod 1980) **n'a pas de suffixe `_toy`** parce qu'il figure comme référence externe, jamais instancié dans le moteur. Il est mentionné en exercice 1.

Cette séparation est sémantique : `_toy` signale au lecteur que le bot est une heuristique pédagogique, pas une reproduction de stratégie publiée.



## Objectifs d'apprentissage

- Verifier que `(D,D)` est l'unique equilibre de Nash du Dilemme du prisonnier classique (Osborne §3.2).
- Definir un `ProgramAgent` borne dont l'action depend de la representation de l'adversaire.
- Implementer cinq bots-`_toy` : `CooperateBot_toy`, `DefectBot_toy`, `FairBot_toy`/`DUPOC`, `CUPOD_toy` et `PrudentBot_toy` (cf. header §0 : `_toy` = heuristique locale au notebook, PAS une reproduction de strategie publiee).
- Produire la matrice de confrontation et rendre visibles trois phenomenes : cooperation
  mutuelle, inexploitation, defection paradoxale.
- Distinguer **verdict CALCULE** (`play`), **verdict par defaut** (`depth`), et **exception de timeout** (`SimulationTimeout`, REPAIR axe 1 c.989).
- Verifier le verdict par **deux organe** : coherence interne du moteur + organe strictement independant (`independent_pair_v2`, REPAIR axe 3 c.989).
- Comparer repetition/Axelrod, preuve formelle (Lean), simulation et transparence.


### Prérequis

- `GameTheory-06` (Evolution of Trust — Axelrod).
- `GameTheory-06c` (Folk theorem des jeux répétés).
- `GameTheory-06d` (statique comparative sympathie vs engagement).


## 1. Le Dilemme du prisonnier classique : `(D,D)` est l'unique équilibre de Nash


On pose la matrice canonique `(T=5, R=3, P=1, S=0)` avec `T > R > P > S` et `2R > T + S`.

| | C | D |
|---|---|---|
| **C** | R, R | S, T |
| **D** | T, S | P, P |

Best response : **D** domine **C** pour chaque joueur, peu importe l'action adverse.
L'unique équilibre de Nash est `(D,D)`.


In [1]:
T, R, P, S = 5, 3, 1, 0
assert T > R > P > S, "Parametres PD non canoniques"
assert 2 * R > T + S, "Cooperation mutuelle > alternance (stricte PD)"

def payoff_self(my_action, other_action):
    if my_action == "C":
        return R if other_action == "C" else S
    return T if other_action == "C" else P

for me in ("C", "D"):
    for other in ("C", "D"):
        print(f"self={me!s:5s} other={other!s:5s} -> {payoff_self(me, other)}")

for my_action in ("C", "D"):
    for other in ("C", "D"):
        assert payoff_self("D", other) >= payoff_self("C", other), (
            f"D ne domine pas C : me={my_action} other={other}"
        )
print("D domine C (PD stricte). Equilibre de Nash unique : (D,D).")


self=C     other=C     -> 3
self=C     other=D     -> 0
self=D     other=C     -> 5
self=D     other=D     -> 1
D domine C (PD stricte). Equilibre de Nash unique : (D,D).


## 2. `ProgramAgent` : un agent qui raisonne sur la **représentation** de l'adversaire


On promeut le jeu sous forme normale en jeu sous forme extensive : chaque joueur est un
programme qui reçoit en argument la **représentation textuelle** (le code) de l'adversaire
et choisit son action. Cette transparence est la **cle** qui change l'issue.

Bornes : on fixe une profondeur de récursion et un budget d'étapes pour éviter la
non-termination. Un agent qui dépasse la borne retourne une action par défaut (`D`).


In [2]:
MAX_DEPTH = 3
STEP_BUDGET = 1000


class SimulationTimeout(Exception):
    """Levee quand le budget step_cap est epuise AVANT calcul du verdict.

    REPAIR c.996 #15175 : SimulationTimeout est une GARDE D'ENTREE, pas une preuve
    de terminaison. La garde decrement avant l'appel au bot, donc si step_cap <= 0
    a l'entree l'exception est levee immediatement (cf. cellule 22 etape 2 pour la
    demonstration). Au-dela, le bot est appele directement : un bot non terminant
    (par ex. `while True: pass`) BLOQUERA le noyau Jupyter -- il n'y a pas
    d'isolation per-cell dans un notebook. La borne `step_cap` borne donc le nombre
    D'APPELS REUSSIS, pas la duree d'un appel qui n'aboutit pas.
    """


def payoff_self(my_action, other_action):
    if my_action == "C":
        return R if other_action == "C" else S
    return T if other_action == "C" else P


def simulate_payoff(player_a, player_b, source_a, source_b,
                    step_cap=STEP_BUDGET, depth=0):
    """Verdict d'une confrontation one-shot, garde d'entree sur step_cap.

    REPAIR c.996 #15175 : la clause docstring precedente affirmait a tort qu'un
    bot non terminant levait SimulationTimeout au plus tard a la deuxieme iteration.
    C'est faux : si step_cap=1 a l'entree, 1-1=0, 0 < 0 est False, le bot est appele
    directement et bloque le noyau s'il ne termine pas. La semantique honnete est :
    SimulationTimeout est levee ssi step_cap <= 0 a l'entree (apres decrement). Le
    test de la cellule 22 etape 2 (step_cap=0 -> exception levee) reste vrai et tient
    la garde. La verite bornee est documentee dans le docstring de SimulationTimeout.

    Returns
    -------
    tuple (action_a, action_b, (payoff_a, payoff_b), status)
        status in {"play", "depth"}. "timeout" -> levee SimulationTimeout.
    """
    step_cap -= 1  # garde d'entree : si <=0 apres decrement, levee avant appel
    if step_cap < 0:
        raise SimulationTimeout(
            f"step_cap epuise avant verdict (step_cap_initial={STEP_BUDGET})"
        )
    if depth >= MAX_DEPTH:
        return ("D", "D",
                (payoff_self("D", "D"), payoff_self("D", "D")),
                "depth")
    action_a = player_a(source_b)
    action_b = player_b(source_a)
    return (
        action_a,
        action_b,
        (payoff_self(action_a, action_b),
         payoff_self(action_b, action_a)),
        "play",  # verdict CALCULE, pas "preuve"
    )


## 3. Cinq bots-programmes


On definit cinq strategies _toy_ (heuristiques locales au notebook) :

- `CooperateBot_toy` : coopere inconditionnellement.
- `DefectBot_toy` : defaille inconditionnellement.
- `FairBot_toy` (analogue a `DUPOC` dans la litterature, **non** une reproduction) : joue `C` si l'adversaire joue `C`, `D` sinon. **Punisseur simple**.
- `CUPOD_toy` (Cooperate Until Provoked Or Defected) : joue `C` tant que l'adversaire joue `C` ou n'a pas encore joue, `D` sinon. **Patient**.
- `PrudentBot_toy` : joue `D` si l'adversaire est un `DefectBot_toy` pur, sinon `C`. **Inspecte avant de decider**.

Le suffixe `_toy` signale que ces bots sont des **heuristiques pedagogiques**, pas des reproductions de strategies publiees (cf. header §0 pour les conventions de nommage et les sources primaires).


In [3]:
def CooperateBot_toy(_other_source):
    return "C"


def DefectBot_toy(_other_source):
    return "D"


def FairBot_toy(other_source):
    if 'return "C"' in other_source:
        return "C"
    return "D"


def CUPOD_toy(other_source):
    if 'return "C"' in other_source:
        return "C"
    return "D"


def PrudentBot_toy(other_source):
    if "DefectBot_toy" in other_source and "CUPOD_toy" not in other_source and "FairBot_toy" not in other_source:
        return "D"
    if 'return "C"' in other_source:
        return "C"
    return "D"


SOURCES = {
    "CooperateBot_toy": 'def CooperateBot_toy(_other_source):\n    return "C"',
    "DefectBot_toy":    'def DefectBot_toy(_other_source):\n    return "D"',
    "FairBot_toy":      'def FairBot_toy(other_source):\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"',
    "CUPOD_toy":        'def CUPOD_toy(other_source):\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"',
    "PrudentBot_toy":   'def PrudentBot_toy(other_source):\n    if "DefectBot_toy" in other_source and "CUPOD_toy" not in other_source and "FairBot_toy" not in other_source:\n        return "D"\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"',
}


## 4. Matrice de confrontation : trois phénomènes


In [4]:
import itertools

BOTS = {
    "CooperateBot_toy": CooperateBot_toy,
    "DefectBot_toy":    DefectBot_toy,
    "FairBot_toy":      FairBot_toy,
    "CUPOD_toy":        CUPOD_toy,
    "PrudentBot_toy":   PrudentBot_toy,
}

rows = []
for name_a, name_b in itertools.product(BOTS, repeat=2):
    try:
        a, b, payoff, status = simulate_payoff(
            BOTS[name_a], BOTS[name_b], SOURCES[name_a], SOURCES[name_b]
        )
    except SimulationTimeout:
        a, b, payoff, status = "D", "D", (1, 1), "timeout"
    rows.append({
        "A": name_a, "B": name_b, "act": f"{a}/{b}",
        "payoff": payoff, "status": status,
    })

print(f"{'A':14s} {'B':14s} {'act':4s} {'payoff_A':>8s} {'payoff_B':>8s} {'status':10s}")
for r in rows:
    print(f"{r['A']:14s} {r['B']:14s} {r['act']:4s} {r['payoff'][0]:>8d} {r['payoff'][1]:>8d} {r['status']:10s}")

assert all(r["status"] == "play" for r in rows), \
    "Toutes les confrontations doivent rendre un verdict 'play' (verdict CALCULE) avec step_cap=1000 et bots deterministes purs"
print(f"25/25 confrontations rendent un verdict 'play' (CALCULE dans la borne).")


A              B              act  payoff_A payoff_B status    
CooperateBot_toy CooperateBot_toy C/C         3        3 play      
CooperateBot_toy DefectBot_toy  C/D         0        5 play      
CooperateBot_toy FairBot_toy    C/C         3        3 play      
CooperateBot_toy CUPOD_toy      C/C         3        3 play      
CooperateBot_toy PrudentBot_toy C/C         3        3 play      
DefectBot_toy  CooperateBot_toy D/C         5        0 play      
DefectBot_toy  DefectBot_toy  D/D         1        1 play      
DefectBot_toy  FairBot_toy    D/D         1        1 play      
DefectBot_toy  CUPOD_toy      D/D         1        1 play      
DefectBot_toy  PrudentBot_toy D/D         1        1 play      
FairBot_toy    CooperateBot_toy C/C         3        3 play      
FairBot_toy    DefectBot_toy  D/D         1        1 play      
FairBot_toy    FairBot_toy    C/C         3        3 play      
FairBot_toy    CUPOD_toy      C/C         3        3 play      
FairBot_toy    PrudentBo

### Lecture de la matrice

Trois phenomenes sont visibles (verdict **calcule** dans la borne step_cap=1000, jamais une preuve formelle) :

1. **Cooperation mutuelle** : `FairBot_toy` face a `CUPOD_toy` ou `CooperateBot_toy` produit `(C, C)`
   avec payoff `(3, 3)`. La transparence du programme permet a chaque bot de detecter la
   clause `return "C"` de l'autre et de choisir `C` en consequence.
2. **Inexploitation** : `DefectBot_toy` face a `CooperateBot_toy` produit `(D, C)` avec payoff
   `(5, 0)`. Le patient (CooperateBot_toy) ne lit rien et se fait exploiter -- l'asymetrie
   informationnelle tue la cooperation meme quand l'adversaire ne joue qu'une fois.
3. **Defection paradoxale** : `PrudentBot_toy` face a `CUPOD_toy` detecte `CUPOD_toy` **et** la clause
   `return "C"` -> il joue `C`. Mais face a `DefectBot_toy` pur, il detecte `DefectBot_toy` sans
   `CUPOD_toy` ni `FairBot_toy` -> il joue `D`. La transparence revele le type de l'adversaire
   et **le patient n'est plus recompense** : seul l'historique de cooperation deja inscrite
   dans le source le sauve.

L'equilibre `(D, D)` classique de la forme normale **n'est plus l'unique issue** dans ce modele :
la transparence du programme est un mecanisme de signal au meme titre que la reputation ou la menace
de represailles (cf. Axelrod & Hamilton 1981 pour la these generale, voir header §0).


## 5. Verificateur indépendant : reproduire la matrice


Pour la rigueur, on **recopie** la matrice depuis un verificateur séparé qui n'importe pas
le moteur principal `simulate_payoff`. Si les deux matrices sont identiques, le résultat
est reproductible hors du moteur.


In [5]:
# Verificateur de coherence INTERNE au moteur (REPAIR axe 3)
# Ce verificateur re-importe BOTS, SOURCES et payoff_self du moteur principal.
# Il sert a confirmer la coherence interne, PAS comme organe independant.
# L'organe strictement independant est `independent_pair_v2` (voir cellule suivante).

def independent_pair(name_a, name_b):
    bot_a = BOTS[name_a]
    bot_b = BOTS[name_b]
    src_a = SOURCES[name_a]
    src_b = SOURCES[name_b]
    act_a = bot_a(src_b)
    act_b = bot_b(src_a)
    return act_a, act_b, payoff_self(act_a, act_b)

ok = 0
for r in rows:
    a2, b2, pay2 = independent_pair(r["A"], r["B"])
    same = (a2, b2) == tuple(r["act"].split("/")) and pay2 == r["payoff"][0]
    assert same, f"Mismatch sur {r['A']} vs {r['B']}: moteur={r['act']}, verificateur_interne={a2}/{b2}"
    ok += 1
print(f"Coherence interne moteur : {ok}/{len(rows)} confrontations agree (trivialement, bots deterministes).")


Coherence interne moteur : 25/25 confrontations agree (trivialement, bots deterministes).


In [6]:
# Second organe declaratif separe (REPAIR c.996 #15175)
# REPAIR c.996 : les 5 fonctions _v2 (version anterieure de cette cellule) recopiaient
# textuellement la logique de string-match du moteur principal, ce qui trahissait la
# promesse d'un organe strictement independant. Remplacement par une **table
# declarative** derivee a la main depuis la specification documentee des bots : on
# re-tape la logique de decision (independamment du code source du moteur) et on
# materialise le verdict attendu en table. La table est lue comme un oracle : si
# le verdict du moteur (cellule 18) coincide avec l'oracle, les deux organes
# -- moteur et oracle -- rendent le meme verdict.


# Specification documentee (independante du code source -- retranscrite a la main) :
#   CooperateBot_toy_v2(other_source) -> "C"                                       inconditionnel
#   DefectBot_toy_v2(other_source)    -> "D"                                       inconditionnel
#   FairBot_toy_v2(other_source)      -> "C" si 'return "C"' dans other_source, sinon "D"
#   CUPOD_toy_v2(other_source)        -> "C" si 'return "C"' dans other_source, sinon "D"
#   PrudentBot_toy_v2(other_source)   -> "D" si "DefectBot_toy_v2" dans other_source
#                                          et "CUPOD_toy_v2"/"FairBot_toy_v2" pas dans other_source
#                                       sinon "C" si 'return "C"' dans other_source, sinon "D"


PAYOFFS_V2 = {
    ("C", "C"): (3, 3),
    ("C", "D"): (0, 5),
    ("D", "C"): (5, 0),
    ("D", "D"): (1, 1),
}


def _spec_action(bot_name, other_name, SOURCES_v2):
    """Spec documentee retranscrite a la main (independante du moteur principal).

    La cle est : la logique de chaque bot _v2 est RETAPEE ici depuis la documentation,
    pas copiee depuis le source. Les memes regles de decision s'appliquent parce que
    c'est le meme algorithme documente, mais l'implementation est separee -- si le
    moteur boguait (par ex. FairBot copiait/collait une mauvaise regex), l'oracle
    continuerait a rendre le bon verdict.
    """
    src = SOURCES_v2[other_name]
    if bot_name == "CooperateBot_toy_v2":
        return "C"
    if bot_name == "DefectBot_toy_v2":
        return "D"
    if bot_name in ("FairBot_toy_v2", "CUPOD_toy_v2"):
        return "C" if 'return "C"' in src else "D"
    if bot_name == "PrudentBot_toy_v2":
        if "DefectBot_toy_v2" in src and "CUPOD_toy_v2" not in src and "FairBot_toy_v2" not in src:
            return "D"
        return "C" if 'return "C"' in src else "D"
    raise ValueError(f"bot inconnu: {bot_name}")


# SOURCES_V2 retranscrit a la main. On utilise des variables separees pour eviter
# les problemes d'echappement de triple-quote avec les chaines multi-lignes.
_SRC_COOPERATE_V2 = 'def CooperateBot_toy_v2(_other_source):\n    return "C"'
_SRC_DEFECT_V2    = 'def DefectBot_toy_v2(_other_source):\n    return "D"'
_SRC_FAIR_V2      = 'def FairBot_toy_v2(other_source):\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"'
_SRC_CUPOD_V2     = 'def CUPOD_toy_v2(other_source):\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"'
_SRC_PRUDENT_V2   = 'def PrudentBot_toy_v2(other_source):\n    if "DefectBot_toy_v2" in other_source and "CUPOD_toy_v2" not in other_source and "FairBot_toy_v2" not in other_source:\n        return "D"\n    if \'return "C"\' in other_source:\n        return "C"\n    return "D"'

SOURCES_V2 = {
    "CooperateBot_toy_v2": _SRC_COOPERATE_V2,
    "DefectBot_toy_v2":    _SRC_DEFECT_V2,
    "FairBot_toy_v2":      _SRC_FAIR_V2,
    "CUPOD_toy_v2":        _SRC_CUPOD_V2,
    "PrudentBot_toy_v2":   _SRC_PRUDENT_V2,
}


def _build_expected_table():
    bots = ["CooperateBot_toy_v2", "DefectBot_toy_v2", "FairBot_toy_v2", "CUPOD_toy_v2", "PrudentBot_toy_v2"]
    table = {}
    for a in bots:
        for b in bots:
            act_a = _spec_action(a, b, SOURCES_V2)
            act_b = _spec_action(b, a, SOURCES_V2)
            table[(a, b)] = (act_a, act_b, PAYOFFS_V2[(act_a, act_b)])
    return table


EXPECTED_TABLE_V2 = _build_expected_table()
assert len(EXPECTED_TABLE_V2) == 25, f"attendu 25 paires, obtenu {len(EXPECTED_TABLE_V2)}"


def independent_pair_v2(name_a, name_b):
    """Verificateur strictement independant (REPAIR c.996).

    Lit le verdict attendu dans la table declarative. La table est construite par
    une fonction de spec RETAPEE A LA MAIN (pas par string-match du moteur), donc
    l'oracle est separe de l'implementation.
    """
    return EXPECTED_TABLE_V2[(name_a, name_b)]


ok_v2 = 0
for r in rows:
    a2, b2, pay2 = independent_pair_v2(
        r["A"].replace("_toy", "_toy_v2"),
        r["B"].replace("_toy", "_toy_v2"),
    )
    same = (a2, b2) == tuple(r["act"].split("/")) and pay2 == r["payoff"]
    assert same, f"Mismatch independent_v2 sur {r['A']} vs {r['B']}: moteur={r['act']}, oracle={a2}/{b2}"
    ok_v2 += 1

assert ok_v2 == 25, f"independent_v2: attendu 25/25, obtenu {ok_v2}/25"
print(f"Matrice reproductible (oracle declaratif v2) : {ok_v2}/25 confrontations agree.")
print("Deux organes de verification : moteur (cellule 18) ET oracle declarative (cette cellule) rendent le meme verdict.")
print(f"Table oracle : {len(EXPECTED_TABLE_V2)} paires, derivee par retranscription manuelle de la spec.")


Matrice reproductible (oracle declaratif v2) : 25/25 confrontations agree.
Deux organes de verification : moteur (cellule 18) ET oracle declarative (cette cellule) rendent le meme verdict.
Table oracle : 25 paires, derivee par retranscription manuelle de la spec.


## 6. Trois états : preuve, absence dans la borne, non-termination


On distingue **trois status**, conformement a l'acceptance REPAIR axe 2 (c.989) :

- **`play`** : la confrontation produit une action et un payoff **calcules** dans la borne
  (`step_cap > 0` et `depth < MAX_DEPTH`). Le verbe "prouver" est reserve aux demonstrations
  formelles ; ici on **calcule** le verdict d'une confrontation one-shot, rien de plus.
- **`depth`** : la profondeur `MAX_DEPTH` est atteinte avant calcul. On retourne l'action
  par defaut `(D, D)` avec payoff `(1, 1)`. Ce n'est PAS une preuve que `(D, D)` est
  l'equilibre -- c'est un verdict par defaut apres atteinte de la profondeur max.
- **`timeout`** : **exception** `SimulationTimeout` levee quand le budget `step_cap` est
  epuise AVANT calcul. REPAIR axe 1 (c.989) : la decrementation de `step_cap` a lieu
  AVANT l'appel au bot, donc un bot non terminant (par ex. `while True: pass`)
  declenche l'exception sans avoir besoin d'etre execute.

Les trois status sont des **verdict d'arret** : `play` est le verdict CALCULE dans la borne,
`depth` et `timeout` sont des **verdict par defaut** apres atteinte de la borne. Aucun des
trois n'est une preuve formelle d'equilibre ; aucun n'est une preuve d'absence d'equilibre
autre que `(D, D)`.


In [7]:
# Demonstration des trois etats via budget explicitement serre.

# (1) depth : on force depth=MAX_DEPTH -> verdict par defaut D/D, status="depth"
a, b, payoff, status = simulate_payoff(
    CooperateBot_toy, CooperateBot_toy, SOURCES["CooperateBot_toy"], SOURCES["CooperateBot_toy"],
    step_cap=STEP_BUDGET, depth=MAX_DEPTH,
)
print(f"depth=MAX_DEPTH -> act={a}/{b}, payoff={payoff}, status={status}")
assert status == "depth", f"attendu depth, obtenu {status}"
assert (a, b) == ("D", "D"), "verdict par defaut doit etre D/D"
assert payoff == (1, 1), "payoff du verdict par defaut doit etre (1, 1)"

# (2) timeout : on force step_cap=0 -> SimulationTimeout LEVEE (REPAIR axe 1)
timeout_leve = False
try:
    a, b, payoff, status = simulate_payoff(
        CooperateBot_toy, CooperateBot_toy, SOURCES["CooperateBot_toy"], SOURCES["CooperateBot_toy"],
        step_cap=0, depth=0,
    )
except SimulationTimeout as e:
    timeout_leve = True
    print(f"step_cap=0 -> SimulationTimeout levee : {e}")
assert timeout_leve, "SimulationTimeout doit etre levee sur step_cap=0 (axe 1 REPAIR)"

# (3) play : verdict CALCULE (pas "preuve") dans la borne
a, b, payoff, status = simulate_payoff(
    CooperateBot_toy, CooperateBot_toy, SOURCES["CooperateBot_toy"], SOURCES["CooperateBot_toy"],
)
print(f"verdict normal -> act={a}/{b}, payoff={payoff}, status={status}")
assert status == "play", f"attendu play (verdict CALCULE), obtenu {status}"
assert (a, b) == ("C", "C"), "CooperateBot_toy vs CooperateBot_toy -> C/C"
assert payoff == (3, 3), "payoff C/C doit etre (3, 3)"

print("Trois etats distincts : 'play' (verdict CALCULE), 'depth' (verdict par defaut borne), 'timeout' (exception).")
print("Aucun n'est une preuve formelle d'equilibre -- tous sont des verdicts d'arret dans la borne.")


depth=MAX_DEPTH -> act=D/D, payoff=(1, 1), status=depth
step_cap=0 -> SimulationTimeout levee : step_cap epuise avant verdict (step_cap_initial=1000)
verdict normal -> act=C/C, payoff=(3, 3), status=play
Trois etats distincts : 'play' (verdict CALCULE), 'depth' (verdict par defaut borne), 'timeout' (exception).
Aucun n'est une preuve formelle d'equilibre -- tous sont des verdicts d'arret dans la borne.


## 7. Comparaison : repetition/Axelrod, preuve bornée, simulation


| Mécanisme | Issue du PD one-shot | Source |
|---|---|---|
| **Forme normale classique** | `(D, D)` unique Nash par dominance stricte | Osborne §3.2 |
| **Répétition (Axelrod 1980)** | `(C, C)` si ombrage du futur (δ ≈ 1) | `GameTheory-06` Evolution of Trust ; Axelrod & Hamilton 1981 |
| **Preuve formelle (Lean)** | `(C, C)` par démonstration dans le calcul des constructions | `GameTheory-06b` Lean |
| **Transparence des programmes** (ce notebook) | `(C, C)` par lecture de la source, dans la borne `step_cap` | Ce notebook -- `GameTheory-06e` |
| **Simulation multi-graines** | distribution empirique | `GameTheory-06c` Folk theorem |

Quatre mécanismes convergent vers `(C, C)` mais par des chemins distincts : la répétition
donne du poids au futur, la preuve formelle établit l'équilibre dans le calcul, la transparence
donne accès à l'intention, la simulation borne l'incertitude. Voir header §0 pour les
références complètes (Shoham-Leyton-Brown §3.4.2 pour la paramétrisation canonique du PD,
Osborne §3.2 pour la dominance stricte, Axelrod & Hamilton 1981 pour la dynamique évolutive,
Critch-Dennis-Russell 2022 §3 pour le cadre d'inspection mutuelle).


## 8. Limites et Open Problems


- **Open Problem 3 de Critch-Dennis-Russell 2022** (cf. arXiv:2208.07006 p. 18) : la conjecture `DUPOC(k)` vs `CUPOD(k)` sur
  l'itération k de l'inspection mutuelle reste **explicitement ouverte** dans la littérature.
  Ce notebook l'illustre mais ne la résout pas -- ne jamais la présenter comme exercice à
  solution attendue. Les bots `_toy` du notebook ne sont **pas** des reproductions de `DUPOC`
  ou `CUPOD` (cf. header §0 : Critch-Dennis-Russell 2022 §3 pour la définition publiée).
- **Bornes fixes** : `MAX_DEPTH = 3` et `STEP_BUDGET = 1000` sont des choix opérationnels.
  Les augmenter peut faire émerger des comportements différents (cf. `GameTheory-06c`
  Folk theorem pour la discussion). REPAIR c.996 #15175 : `step_cap` est une **garde d'entrée**
  (cf. cellule 9 et cellule 22 étape 2). Si `step_cap <= 0` à l'entrée, `SimulationTimeout`
  est levée immédiatement. Au-delà, le bot est appelé directement et un bot non terminant
  bloquera le noyau Jupyter -- il n'y a pas d'isolation per-cell dans un notebook.
- **Inspection par chaîne textuelle** : `PrudentBot_toy` inspecte par `in` Python. Un adversaire
  qui obfusque sa source (ex : `chr(67)` au lieu de `"C"`) trompe l'inspection. Voir
  `GameTheory-06b` Lean pour la formalisation, et Critch-Dennis-Russell 2022 §3 (cadre
  d'inspection mutuelle de programmes) pour la discussion publiée de ces mécanismes.


## Exercices


### Exercice 1 — Bot inédit : `TitForTatBot`

Implémentez `TitForTatBot` qui joue `C` au premier tour et recopie l'action de l'adversaire
au tour suivant. Vous aurez besoin de **stocker l'historique** entre deux appels : la signature
de la fonction `Callable[[str], str]` ne suffit pas. Étendez `simulate_payoff` (ou créez une
variante `simulate_repeated`) qui passe l'historique comme argument supplémentaire.

Indice : la transparence ne s'applique plus au même jeu — `TitForTat` est une stratégie
des jeux répétés. Vous pouvez combiner les deux notebooks (`GameTheory-06` et `06e`) pour
tester votre variante.


### Exercice 2 — Contre-exemple de robustesse

Trouvez un bot `X` tel que `FairBot` face à `X` **n'obtient pas** `(C, C)`. Justifiez en
montrant la chaîne textuelle que `FairBot` inspecte et pourquoi elle est silencieuse.

Indice : un bot qui retourne `C` conditionnellement à un **autre** signal que la présence
de `return "C"` peut déjouer l'inspection par chaîne.


### Exercice 3 — Variation de borne

Augmentez `MAX_DEPTH` de 3 à 5. La matrice change-t-elle ? Si oui, identifiez le bot qui
profite de la profondeur accrue et expliquez pourquoi. Si non, justifiez l'invariance.

Indice : avec une profondeur suffisante, `PrudentBot` peut itérer sur sa propre décision
avant de choisir. La question est de savoir si cette auto-inspection est bornée.
